# HGTCM P1 - Preprocessing and Line Assignment

This notebook combines target page preprocessing, first-pass projection assignment, page-wise exact-count crop selection, and final target manifest generation.

Flow: raw paired pages -> cleaned black-text-on-white pages -> base assignment -> page-wise decision improvement -> final line crops and manifest for P2.

Exact count is required. Near-miss pages stay rejected.

## Imports


In [1]:
# ## Imports: stdlib + OpenCV, NumPy, pandas; PIL; small helpers for tables in the UI.
import csv
import json
import math
import os
import random
import re
import shutil
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageOps
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 140)


def summary_table(payload: dict, title: str | None = None):
    if title:
        display(Markdown(f"### {title}"))
    display(pd.DataFrame([payload]))


def display_paths(paths: list[Path | str], title: str | None = None):
    if title:
        display(Markdown(f"### {title}"))
    display(pd.DataFrame([{"file": Path(path).name, "path": str(path)} for path in paths]))


## Run Configuration


In [2]:
# ## Run configuration: env + paths (derived pages, line crops, this run's results id).
PROJECT_ROOT = Path(os.environ.get("PROJECT_DECODE_ROOT", ".")).resolve()
DATA_RUN_LABEL = os.environ.get("DATA_RUN_LABEL", "affine_edge_full").strip() or "affine_edge_full"
BASE_RUN_LABEL = os.environ.get("P1_BASE_RUN_LABEL", f"{DATA_RUN_LABEL}_base").strip() or f"{DATA_RUN_LABEL}_base"
CLEANED_PAGE_RUN_LABEL = os.environ.get("P1_CLEANED_PAGE_RUN_LABEL", "projection_v1_affine_full").strip() or "projection_v1_affine_full"
P1_RUN_ID = os.environ.get("P1_RUN_ID", f"p1_preprocessing_assignment_{DATA_RUN_LABEL}").strip() or f"p1_preprocessing_assignment_{DATA_RUN_LABEL}"
P1_OUTPUT_DIR = PROJECT_ROOT / "results" / P1_RUN_ID
TARGET_ACCEPTED_PAGES = int(os.environ.get("TARGET_ACCEPTED_PAGES", "0"))
MAX_SCAN_PAGES = int(os.environ.get("MAX_SCAN_PAGES", "0"))
P1_ITERATIONS = max(1, int(os.environ.get("P1_ITERATIONS", "2")))
SEED = int(os.environ.get("SEED", "42"))
QA_PROFILE = os.environ.get("QA_PROFILE", "strict").strip() or "strict"
CLAHE_CLIP_LIMIT = float(os.environ.get("CLAHE_CLIP_LIMIT", "2.0"))
CLAHE_TILE_GRID_SIZE = int(os.environ.get("CLAHE_TILE_GRID_SIZE", "8"))
P1_DEBUG_LIMIT = int(os.environ.get("P1_DEBUG_LIMIT", "30"))
FINAL_CROP_PAD_Y_PX = int(os.environ.get("FINAL_CROP_PAD_Y_PX", "4"))
EDGE_DECONTAMINATION_ENABLED = os.environ.get("EDGE_DECONTAMINATION_ENABLED", "1").strip().lower() in {"1", "true", "yes", "y"}
PAGE_AFFINE_DESKEW_ENABLED = os.environ.get("PAGE_AFFINE_DESKEW_ENABLED", "1").strip().lower() in {"1", "true", "yes", "y"}
PAGE_AFFINE_DESKEW_MAX_DEG = float(os.environ.get("PAGE_AFFINE_DESKEW_MAX_DEG", "3.0"))
LINE_STRAIGHTENING_ENABLED = os.environ.get("LINE_STRAIGHTENING_ENABLED", "0").strip().lower() in {"1", "true", "yes", "y"}
LINE_STRAIGHTENING_MIN_DEG = float(os.environ.get("LINE_STRAIGHTENING_MIN_DEG", "0.15"))
LINE_STRAIGHTENING_MAX_DEG = float(os.environ.get("LINE_STRAIGHTENING_MAX_DEG", "2.0"))
REFRESH_CLEANED_PAGES = os.environ.get("REFRESH_CLEANED_PAGES", "1").strip().lower() in {"1", "true", "yes", "y"}

DATA_DIR = PROJECT_ROOT / "data"
ZIPPED_SOURCES_DIR = DATA_DIR / "zipped sources"
MANIFESTS_DIR = PROJECT_ROOT / "manifests"
EXTRACTED_PAGES_DIR = DATA_DIR / "extracted" / "target_pages"
EXTRACTED_GT_DIR = DATA_DIR / "extracted" / "target_gt"
CLEANED_PAGE_DIR = DATA_DIR / "derived" / "target_cleaned_pages" / CLEANED_PAGE_RUN_LABEL
BASE_CROP_DIR = DATA_DIR / "derived" / "target_line_crops" / BASE_RUN_LABEL
FINAL_CROP_DIR = DATA_DIR / "derived" / "target_line_crops" / DATA_RUN_LABEL
PREPROCESSING_METADATA_DIR = DATA_DIR / "derived" / "target_preprocessing_metadata" / DATA_RUN_LABEL

for path in [P1_OUTPUT_DIR, MANIFESTS_DIR, EXTRACTED_PAGES_DIR, EXTRACTED_GT_DIR, CLEANED_PAGE_DIR, BASE_CROP_DIR, FINAL_CROP_DIR, PREPROCESSING_METADATA_DIR]:
    path.mkdir(parents=True, exist_ok=True)

summary_table({
    "project_root": str(PROJECT_ROOT),
    "data_run_label": DATA_RUN_LABEL,
    "base_run_label": BASE_RUN_LABEL,
    "cleaned_page_run_label": CLEANED_PAGE_RUN_LABEL,
    "p1_run_id": P1_RUN_ID,
    "target_accepted_pages": TARGET_ACCEPTED_PAGES or "all",
    "max_scan_pages": MAX_SCAN_PAGES or "all paired pages",
    "iterations": P1_ITERATIONS,
    "refresh_cleaned_pages": REFRESH_CLEANED_PAGES,
    "qa_profile": QA_PROFILE,
}, "P1 configuration")


### P1 configuration

,project_root,data_run_label,base_run_label,cleaned_page_run_label,p1_run_id,target_accepted_pages,max_scan_pages,iterations,refresh_cleaned_pages,qa_profile
0,C:\Users\Jobias\OneDrive\Desktop\Project Decode,affine_edge_full,affine_edge_full_base,projection_v1_affine_full,p1_preprocessing_assignment_affine_edge_full,all,all paired pages,2,True,strict


## IO Helpers


In [3]:
# ## I/O and text helpers: JSON/JSONL, safe clear only under data/derived, normalize.
def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def clear_generated_dir(path: Path) -> Path:
    path = path.resolve()
    allowed = (DATA_DIR / "derived").resolve()
    if allowed not in path.parents and path != allowed:
        raise ValueError(f"Refusing to clear outside data/derived: {path}")
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def write_json(path: Path, payload: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


def write_jsonl(path: Path, rows: list[dict]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def normalize_text(text: str) -> str:
    return " ".join((text or "").replace("\r\n", "\n").replace("\r", "\n").split())


def nonempty_lines(text: str) -> list[str]:
    return [line.strip() for line in (text or "").splitlines() if line.strip()]


def page_number_from_name(name: str) -> int | None:
    match = re.search(r"page[_-]?(\d+)", name, re.IGNORECASE)
    return int(match.group(1)) if match else None


def image_zip_paths() -> list[Path]:
    return sorted(ZIPPED_SOURCES_DIR.glob("drive-download*.zip"))


def gt_zip_paths() -> list[Path]:
    return sorted(ZIPPED_SOURCES_DIR.glob("GroundTruth*.zip"))


def index_image_members() -> dict[int, tuple[Path, str]]:
    index = {}
    for zip_path in image_zip_paths():
        with zipfile.ZipFile(zip_path) as archive:
            for member in archive.namelist():
                if not member.lower().endswith((".tif", ".tiff", ".png", ".jpg", ".jpeg")):
                    continue
                page_no = page_number_from_name(member)
                if page_no is not None:
                    index.setdefault(page_no, (zip_path, member))
    return index


def index_gt_members() -> dict[int, tuple[Path, str]]:
    index = {}
    for zip_path in gt_zip_paths():
        with zipfile.ZipFile(zip_path) as archive:
            for member in archive.namelist():
                if not member.lower().endswith(".txt"):
                    continue
                page_no = page_number_from_name(member)
                if page_no is not None:
                    index.setdefault(page_no, (zip_path, member))
    return index


def extract_zip_member(zip_path: Path, member: str, output_path: Path) -> Path:
    if output_path.exists():
        return output_path
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        with archive.open(member) as src, output_path.open("wb") as dst:
            shutil.copyfileobj(src, dst)
    return output_path


def read_gt_text(zip_path: Path, member: str, output_path: Path) -> str:
    if output_path.exists():
        return output_path.read_text(encoding="utf-8", errors="replace")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        text = archive.read(member).decode("utf-8", errors="replace")
    output_path.write_text(text, encoding="utf-8")
    return text


## Input Inventory


In [4]:
# ## Zip inventory: page numbers present in image zips vs GT; write input_inventory.json.
image_index = index_image_members()
gt_index = index_gt_members()
paired_page_numbers = sorted(set(image_index) & set(gt_index))
if MAX_SCAN_PAGES > 0:
    paired_page_numbers = paired_page_numbers[:MAX_SCAN_PAGES]

inventory = {
    "image_zip_count": len(image_zip_paths()),
    "gt_zip_count": len(gt_zip_paths()),
    "image_pages": len(image_index),
    "ground_truth_pages": len(gt_index),
    "paired_pages_selected": len(paired_page_numbers),
    "image_only_pages": len(set(image_index) - set(gt_index)),
    "gt_only_pages": len(set(gt_index) - set(image_index)),
}
write_json(P1_OUTPUT_DIR / "input_inventory.json", inventory)
summary_table(inventory, "Input inventory")


### Input inventory

,image_zip_count,gt_zip_count,image_pages,ground_truth_pages,paired_pages_selected,image_only_pages,gt_only_pages
0,11,1,785,811,784,1,27


## Page Preprocessing


In [5]:
# ## Page geometry: margin crop, global skew, CLAHE, affine deskew, binary for projection.
# Why: deskew + contrast normalization reduce projection noise from scan slant and illumination drift.
def crop_page_margins(gray: np.ndarray, padding: int = 24) -> tuple[np.ndarray, dict]:
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, ink = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)
    coords = np.column_stack(np.where(ink > 0))
    h, w = gray.shape[:2]
    if coords.size == 0:
        return gray, {"left": 0, "top": 0, "right": w, "bottom": h, "status": "no_ink_found"}
    top, left = coords.min(axis=0)
    bottom, right = coords.max(axis=0) + 1
    left = max(0, int(left) - padding)
    top = max(0, int(top) - padding)
    right = min(w, int(right) + padding)
    bottom = min(h, int(bottom) + padding)
    return gray[top:bottom, left:right], {"left": left, "top": top, "right": right, "bottom": bottom, "status": "ok"}


def estimate_page_skew_angle(gray: np.ndarray) -> tuple[float, float]:
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, ink = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)
    coords = np.column_stack(np.where(ink > 0))
    if len(coords) < 500:
        return 0.0, 0.0
    # Use a downsampled point cloud so minAreaRect stays fast and robust enough for global page deskew.
    if len(coords) > 8000:
        step = max(1, len(coords) // 8000)
        coords = coords[::step]
    points = coords[:, ::-1].astype("float32")  # x, y
    rect = cv2.minAreaRect(points)
    angle = float(rect[-1])
    if angle < -45.0:
        angle += 90.0
    if angle > 45.0:
        angle -= 90.0
    confidence = min(1.0, len(coords) / 8000.0)
    return angle, confidence


def apply_affine_page_deskew(gray: np.ndarray) -> tuple[np.ndarray, dict]:
    angle, confidence = estimate_page_skew_angle(gray)
    meta = {"enabled": PAGE_AFFINE_DESKEW_ENABLED, "angle_deg": angle, "confidence": confidence, "applied": False}
    if not PAGE_AFFINE_DESKEW_ENABLED:
        return gray, meta
    if abs(angle) < 0.15 or abs(angle) > PAGE_AFFINE_DESKEW_MAX_DEG or confidence < 0.35:
        meta["reason"] = "angle_or_confidence_outside_guardrails"
        return gray, meta
    height, width = gray.shape[:2]
    center = (width / 2.0, height / 2.0)
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    cos = abs(float(matrix[0, 0]))
    sin = abs(float(matrix[0, 1]))
    expanded_width = int(math.ceil((height * sin) + (width * cos)))
    expanded_height = int(math.ceil((height * cos) + (width * sin)))
    matrix[0, 2] += (expanded_width / 2.0) - center[0]
    matrix[1, 2] += (expanded_height / 2.0) - center[1]
    rotated = cv2.warpAffine(
        gray,
        matrix,
        (expanded_width, expanded_height),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=255,
    )
    recropped, recrop_box = crop_page_margins(rotated, padding=24)
    meta["applied"] = True
    meta["matrix"] = matrix.tolist()
    meta["input_size"] = [int(width), int(height)]
    meta["expanded_size"] = [int(expanded_width), int(expanded_height)]
    meta["recrop_box_after_rotation"] = recrop_box
    meta["padding_strategy"] = "expanded_canvas_then_margin_recrop"
    return recropped, meta


def apply_clahe(gray: np.ndarray) -> np.ndarray:
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT, tileGridSize=(CLAHE_TILE_GRID_SIZE, CLAHE_TILE_GRID_SIZE))
    return clahe.apply(gray)


def preprocess_page(raw_page_path: Path, cleaned_page_path: Path, metadata_path: Path) -> dict:
    if cleaned_page_path.exists() and not REFRESH_CLEANED_PAGES:
        return json.loads(metadata_path.read_text(encoding="utf-8")) if metadata_path.exists() else {"status": "reused"}
    image = Image.open(raw_page_path).convert("L")
    gray = np.array(ImageOps.exif_transpose(image))
    cropped, margin_box = crop_page_margins(gray)
    cropped, deskew_meta = apply_affine_page_deskew(cropped)
    contrast = apply_clahe(cropped)
    blur = cv2.GaussianBlur(contrast, (5, 5), 0)
    _, bw = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    if float(bw.mean()) < 127.0:
        bw = 255 - bw
    cleaned_page_path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(bw).save(cleaned_page_path)
    metadata = {
        "status": "ok",
        "input_path": str(raw_page_path),
        "output_path": str(cleaned_page_path),
        "margin_crop": margin_box,
        "page_affine_deskew": deskew_meta,
        "preprocess_order": ["crop_margins", "optional_page_affine_deskew", "grayscale", "clahe", "gaussian_blur", "otsu_black_text_on_white"],
        "clahe_clip_limit": CLAHE_CLIP_LIMIT,
        "clahe_tile_grid_size": CLAHE_TILE_GRID_SIZE,
        "cleaned_size": list(Image.open(cleaned_page_path).size),
    }
    write_json(metadata_path, metadata)
    return metadata


def binary_from_cleaned_page(cleaned_page_path: Path) -> np.ndarray:
    gray = np.array(Image.open(cleaned_page_path).convert("L"))
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)
    return binary


## Projection Profiles


In [6]:
# ## Projection search space: baseline + grid of threshold, max height, padding, gap by iteration.
# Why: a small hyperparameter grid is cheaper than full search and captures most line-height/layout variance.
BASELINE_CONFIG = {
    "name": "baseline_projection",
    "projection_threshold": 0.15,
    "morphology_kernel_width": 0,
    "min_line_height": 20,
    "max_line_height": 80,
    "gap_merge": 0,
    "padding_y_px": 5,
    "padding_y_frac": 0.00,
    "min_crop_height": 15,
}


def candidate_configs(iteration: int) -> list[dict]:
    thresholds = [0.08, 0.10, 0.12, 0.15, 0.18]
    max_heights = [80, 95, 110]
    paddings = [6, 8, 10]
    gap_merges = [0, 2]
    if iteration >= 2:
        thresholds = [0.06, 0.07] + thresholds + [0.20, 0.22]
        max_heights = max_heights + [125]
        paddings = paddings + [12]
    if iteration >= 3:
        thresholds = [0.05] + thresholds + [0.24]
        max_heights = max_heights + [140]
    configs = [BASELINE_CONFIG]
    for threshold in thresholds:
        for max_height in max_heights:
            for padding in paddings:
                for gap in gap_merges:
                    configs.append({
                        "name": f"iter{iteration}_thr{threshold:.2f}_max{max_height}_pad{padding}_gap{gap}",
                        "projection_threshold": threshold,
                        "morphology_kernel_width": 0,
                        "min_line_height": 16,
                        "max_line_height": max_height,
                        "gap_merge": gap,
                        "padding_y_px": padding,
                        "padding_y_frac": 0.12,
                        "min_crop_height": 18,
                    })
    return configs


## Projection Detector


In [7]:
# ## Binary page: suppress horizontal rules, merge spans, de-overlap, build line boxes.
# Why: horizontal-rule suppression limits false positives from table borders and separators.
def suppress_horizontal_rules(binary: np.ndarray) -> np.ndarray:
    cleaned = binary.copy()
    if cleaned.size == 0:
        return cleaned
    height = cleaned.shape[0]
    row_ink_fraction = np.mean(cleaned > 0, axis=1)
    rule_rows = np.where(row_ink_fraction > 0.62)[0]
    for row in rule_rows:
        cleaned[max(0, int(row) - 1):min(height, int(row) + 2), :] = 0
    return cleaned


def merge_spans(spans: list[tuple[int, int]], gap_merge: int) -> list[tuple[int, int]]:
    if not spans:
        return []
    merged = [list(spans[0])]
    for top, bottom in spans[1:]:
        if top - merged[-1][1] <= gap_merge:
            merged[-1][1] = bottom
        else:
            merged.append([top, bottom])
    return [(int(top), int(bottom)) for top, bottom in merged]


def deoverlap_boxes(boxes: list[tuple[int, int, int, int]], height: int) -> list[tuple[int, int, int, int]]:
    if len(boxes) < 2:
        return boxes
    boxes = sorted(boxes, key=lambda item: item[1])
    fixed = []
    for idx, (left, top, right, bottom) in enumerate(boxes):
        if idx > 0:
            top = max(top, (boxes[idx - 1][3] + top) // 2)
        if idx < len(boxes) - 1:
            bottom = min(bottom, (bottom + boxes[idx + 1][1]) // 2)
        top = max(0, min(height - 1, top))
        bottom = max(top + 1, min(height, bottom))
        fixed.append((left, top, right, bottom))
    return fixed


def detect_boxes(binary: np.ndarray, cfg: dict) -> list[tuple[int, int, int, int]]:
    height, width = binary.shape[:2]
    work = suppress_horizontal_rules(binary)
    projection = np.sum(work > 0, axis=1).astype("float32")
    if projection.size == 0 or float(projection.max()) <= 0:
        return []
    mask = (projection / float(projection.max())) > float(cfg["projection_threshold"])
    spans = []
    start = None
    min_line_height = int(cfg["min_line_height"])
    for idx, value in enumerate(mask):
        if value and start is None:
            start = idx
        elif not value and start is not None:
            if idx - start >= min_line_height:
                spans.append((start, idx))
            start = None
    if start is not None and len(mask) - start >= min_line_height:
        spans.append((start, len(mask)))
    spans = merge_spans(spans, int(cfg["gap_merge"]))
    boxes = []
    for top, bottom in spans:
        raw_height = bottom - top
        if raw_height < min_line_height or raw_height > int(cfg["max_line_height"]):
            continue
        pad = max(int(cfg["padding_y_px"]), int(round(raw_height * float(cfg["padding_y_frac"]))))
        y1 = max(0, top - pad)
        y2 = min(height, bottom + pad)
        if y2 - y1 >= int(cfg["min_crop_height"]):
            boxes.append((0, y1, width, y2))
    return deoverlap_boxes(boxes, height)


def box_quality_penalty(binary: np.ndarray, boxes: list[tuple[int, int, int, int]]) -> float:
    if not boxes:
        return 999.0
    penalties = []
    for left, top, right, bottom in boxes:
        crop = binary[top:bottom, left:right] > 0
        if crop.size == 0:
            penalties.append(3.0)
            continue
        height = max(1, crop.shape[0])
        edge_rows = max(1, min(4, height // 5))
        penalty = 0.0
        if float(crop[:edge_rows, :].mean()) > 0.025:
            penalty += 0.45
        if float(crop[-edge_rows:, :].mean()) > 0.025:
            penalty += 0.45
        if float(crop.mean()) < 0.002:
            penalty += 1.0
        column_ink = crop.sum(axis=0)
        streak_fraction = float((column_ink > height * 0.55).mean()) if column_ink.size else 0.0
        if streak_fraction > 0.02:
            penalty += min(2.0, streak_fraction * 20.0)
        if height > 150:
            penalty += 0.75
        penalties.append(penalty)
    return float(np.mean(penalties))


## Count Repair Helpers


In [8]:
# ## Attempt records: split boxes, score, exact_attempt when detected count == GT line count.
# Why: exact count matching enforces alignment with GT lines and avoids label-shift during training.
def split_box(box: list[int], pieces: int) -> list[list[int]]:
    left, top, right, bottom = map(int, box)
    height = bottom - top
    if pieces <= 1 or height < pieces * 12:
        return [box]
    cuts = [top + round(height * idx / pieces) for idx in range(pieces + 1)]
    return [[left, int(cuts[idx]), right, int(cuts[idx + 1])] for idx in range(pieces)]


def attempt_from_boxes(page: dict, cfg: dict, boxes: list[tuple[int, int, int, int]], binary: np.ndarray, method: str, decision_pass: int, notes: list[str] | None = None) -> dict:
    gt_count = int(page["ground_truth_line_count"])
    return {
        "page_id": page["page_id"],
        "method": method,
        "decision_pass": decision_pass,
        "config_name": cfg.get("name", method),
        "config": {key: value for key, value in cfg.items() if key != "name"},
        "detected_line_count": len(boxes),
        "ground_truth_line_count": gt_count,
        "line_count_delta": len(boxes) - gt_count,
        "quality_penalty": box_quality_penalty(binary, boxes) if boxes else 999.0,
        "boxes": [list(map(int, box)) for box in boxes],
        "notes": notes or [],
    }


def exact_attempt(attempts: list[dict]) -> dict | None:
    exact = [attempt for attempt in attempts if attempt["line_count_delta"] == 0]
    return sorted(exact, key=lambda row: (row["quality_penalty"], row["config_name"]))[0] if exact else None


def best_near_attempt(attempts: list[dict]) -> dict:
    return sorted(attempts, key=lambda row: (abs(row["line_count_delta"]), row["quality_penalty"], row["config_name"]))[0]


def undercount_repair_attempt(page: dict, carried: dict, binary: np.ndarray) -> dict | None:
    missing = -int(carried["line_count_delta"])
    if missing not in {1, 2, 3}:
        return None
    boxes = [list(map(int, box)) for box in carried.get("boxes", [])]
    if not boxes:
        return None
    heights = [box[3] - box[1] for box in boxes]
    median_height = float(np.median(heights)) if heights else 0.0
    candidates = []
    for idx, box in enumerate(boxes):
        height_score = (box[3] - box[1]) / max(1.0, median_height)
        if height_score >= 1.35:
            candidates.append((height_score, "tall_band", idx))
    for idx in range(len(boxes) - 1):
        gap = boxes[idx + 1][1] - boxes[idx][3]
        if gap >= max(10, median_height * 0.45):
            candidates.append((float(gap), "gap", idx))
    candidates = sorted(candidates, reverse=True)[:6]
    attempts = []
    for _, kind, idx in candidates:
        repaired = [list(box) for box in boxes]
        if kind == "tall_band":
            repaired = repaired[:idx] + split_box(repaired[idx], missing + 1) + repaired[idx + 1:]
        elif kind == "gap":
            top = boxes[idx][3]
            bottom = boxes[idx + 1][1]
            if bottom - top < 10 or missing > 1:
                continue
            repaired = repaired[: idx + 1] + [[0, top, binary.shape[1], bottom]] + repaired[idx + 1:]
        repaired = sorted(repaired, key=lambda box: box[1])
        attempts.append(attempt_from_boxes(page, {"name": carried["config_name"], **carried.get("config", {})}, [tuple(box) for box in repaired], binary, "undercount_repair", 3, [f"repair_kind={kind}", f"missing={missing}"]))
    return exact_attempt(attempts)


def box_is_structural_noise(binary: np.ndarray, box: list[int]) -> bool:
    left, top, right, bottom = map(int, box)
    crop = binary[top:bottom, left:right] > 0
    if crop.size == 0:
        return True
    height, width = crop.shape[:2]
    if height <= 4:
        return True
    ink_density = float(crop.mean())
    row_fill = float((crop.mean(axis=1) > 0.55).mean())
    component_count, _, _, _ = cv2.connectedComponentsWithStats(crop.astype("uint8"), connectivity=8)
    components = max(0, component_count - 1)
    return bool((width > 350 and height <= 12 and row_fill > 0.15) or (width > 350 and ink_density > 0.42 and height <= 18) or (components <= 1 and ink_density < 0.01))


def overcount_cleanup_attempt(page: dict, carried: dict, binary: np.ndarray) -> dict | None:
    extra = int(carried["line_count_delta"])
    if extra < 4:
        return None
    boxes = [list(map(int, box)) for box in carried.get("boxes", [])]
    removable = [idx for idx, box in enumerate(boxes) if box_is_structural_noise(binary, box)]
    if len(removable) < extra:
        return None
    remove_set = set(removable[:extra])
    cleaned = [box for idx, box in enumerate(boxes) if idx not in remove_set]
    attempt = attempt_from_boxes(page, {"name": carried["config_name"], **carried.get("config", {})}, [tuple(box) for box in cleaned], binary, "overcount_cleanup", 4, [f"removed_structural_noise_boxes={sorted(remove_set)}"])
    return attempt if attempt["line_count_delta"] == 0 else None


## Crop Cleanup And QA


In [9]:
# ## Per-line crops: QA flags, padding cleanup, line straighten, write PNG; then page train/val/test split.
# Why: QC flags and optional line straightening improve supervision quality without mutating text labels.

def crop_quality_audit(crop_path: Path, normalized_text: str) -> dict:
    gray = np.array(Image.open(crop_path).convert("L"))
    _, binary = cv2.threshold(cv2.GaussianBlur(gray, (3, 3), 0), 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)
    ink = binary > 0
    h, w = ink.shape[:2]
    reasons = []
    if h < 12:
        reasons.append("crop_too_short")
    if h > 180:
        reasons.append("crop_too_tall_for_single_line")
    if float(ink.mean()) < 0.0015 and normalized_text:
        reasons.append("blank_or_low_ink_crop")
    edge = max(1, min(4, h // 5))
    if float(ink[:edge, :].mean()) > 0.09:
        reasons.append("ink_touches_top_border")
    if float(ink[-edge:, :].mean()) > 0.09:
        reasons.append("ink_touches_bottom_border")
    col_ink = ink.sum(axis=0)
    if col_ink.size and float(ink.mean()) > 0.01 and float((col_ink > h * 0.65).mean()) > 0.12:
        reasons.append("barcode_like_vertical_streak_crop")
    component_count, _, _, _ = cv2.connectedComponentsWithStats(ink.astype("uint8"), connectivity=8)
    components = max(0, component_count - 1)
    if len(normalized_text) >= 20 and components < max(2, len(normalized_text) // 14):
        reasons.append("too_few_character_components_for_label")
    return {"status": "ok" if not reasons else "flagged", "reasons": reasons, "image_size": [int(w), int(h)], "ink_fraction": float(ink.mean()), "component_count": int(components)}


# ## decontaminate_crop_edges: remove ink in padding that does not touch the line body.

def decontaminate_crop_edges(crop: Image.Image, pad_y: int) -> tuple[Image.Image, dict]:
    """Remove only disconnected ink in the final added top/bottom padding bands."""
    if not EDGE_DECONTAMINATION_ENABLED or pad_y <= 0:
        return crop, {"enabled": EDGE_DECONTAMINATION_ENABLED, "removed_components": 0, "removed_pixels": 0}
    arr = np.array(crop.convert("RGB"))
    height, width = arr.shape[:2]
    if height <= pad_y * 2 + 2:
        return crop, {"enabled": True, "removed_components": 0, "removed_pixels": 0, "reason": "crop_too_short"}
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    _, binary = cv2.threshold(cv2.GaussianBlur(gray, (3, 3), 0), 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)
    edge_mask = np.zeros((height, width), dtype=bool)
    edge_mask[:pad_y, :] = True
    edge_mask[height - pad_y:, :] = True
    core_mask = np.zeros((height, width), dtype=bool)
    core_mask[pad_y:height - pad_y, :] = True
    component_count, labels, stats, _ = cv2.connectedComponentsWithStats((binary > 0).astype("uint8"), connectivity=8)
    remove_mask = np.zeros((height, width), dtype=bool)
    removed_components = 0
    for label in range(1, component_count):
        component = labels == label
        if not component.any():
            continue
        # Keep anything that reaches the original selected line box. This protects real ascenders, descenders, and dots.
        if bool((component & core_mask).any()):
            continue
        if not bool((component & edge_mask).any()):
            continue
        area = int(stats[label, cv2.CC_STAT_AREA])
        if area <= 1:
            continue
        remove_mask |= component
        removed_components += 1
    removed_pixels = int(remove_mask.sum())
    if removed_pixels:
        arr[remove_mask] = 255
        # Feather only the edited edge pixels so the crop does not gain hard digital scars.
        feather = cv2.dilate(remove_mask.astype("uint8"), np.ones((3, 3), dtype="uint8"), iterations=1).astype(bool)
        blurred = cv2.GaussianBlur(arr, (3, 3), 0)
        arr[feather & edge_mask] = blurred[feather & edge_mask]
    return Image.fromarray(arr), {"enabled": True, "removed_components": removed_components, "removed_pixels": removed_pixels, "pad_y": pad_y}


# ## _line_ink_mask: shared binarized ink for line skew + straighten.

def _line_ink_mask(gray: np.ndarray) -> np.ndarray:
    if gray.ndim == 3:
        gray = cv2.cvtColor(gray, cv2.COLOR_RGB2GRAY)
    working = gray.astype("uint8")
    if float(working.mean()) < 127.0:
        working = 255 - working
    blur = cv2.GaussianBlur(working, (3, 3), 0)
    _, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)
    return binary > 0


def estimate_line_skew_angle(gray: np.ndarray) -> tuple[float, float]:
    ink = _line_ink_mask(gray)
    ys, xs = np.where(ink)
    if xs.size < 40:
        return 0.0, 0.0
    x_span = int(xs.max() - xs.min() + 1)
    y_span = int(ys.max() - ys.min() + 1)
    if x_span < max(48, y_span * 3):
        return 0.0, 0.0
    coords = np.column_stack([xs.astype("float32"), ys.astype("float32")])
    centered = coords - coords.mean(axis=0, keepdims=True)
    cov = np.cov(centered, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)
    major_val = float(eigvals[order[-1]])
    minor_val = max(float(eigvals[order[0]]), 1e-6)
    ratio = major_val / minor_val
    if ratio < 4.0:
        return 0.0, min(1.0, ratio / 4.0)
    major = eigvecs[:, order[-1]]
    angle = math.degrees(math.atan2(float(major[1]), float(major[0])))
    angle = ((angle + 90.0) % 180.0) - 90.0
    if angle > 45.0:
        angle -= 90.0
    if angle < -45.0:
        angle += 90.0
    confidence = min(1.0, ratio / 14.0)
    return float(angle), float(confidence)


def _line_straightening_score(gray: np.ndarray) -> float:
    ink = _line_ink_mask(gray)
    h, w = ink.shape[:2]
    if h == 0 or w == 0 or not ink.any():
        return 999.0
    active_rows = ink.sum(axis=1) > 0
    edge = max(1, min(4, h // 5))
    edge_penalty = float(ink[:edge, :].mean() + ink[-edge:, :].mean()) * 8.0
    height_penalty = max(0.0, (h - 120) / 120.0)
    return float(active_rows.mean()) + edge_penalty + height_penalty


def _rotate_expand_white(gray: np.ndarray, angle: float) -> np.ndarray:
    h, w = gray.shape[:2]
    pad_y = max(8, int(round(h * 0.18)))
    pad_x = max(8, int(round(w * 0.01)))
    padded = cv2.copyMakeBorder(gray, pad_y, pad_y, pad_x, pad_x, cv2.BORDER_CONSTANT, value=255)
    ph, pw = padded.shape[:2]
    center = (pw / 2.0, ph / 2.0)
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    cos = abs(float(matrix[0, 0]))
    sin = abs(float(matrix[0, 1]))
    expanded_width = int(math.ceil((ph * sin) + (pw * cos)))
    expanded_height = int(math.ceil((ph * cos) + (pw * sin)))
    matrix[0, 2] += (expanded_width / 2.0) - center[0]
    matrix[1, 2] += (expanded_height / 2.0) - center[1]
    return cv2.warpAffine(padded, matrix, (expanded_width, expanded_height), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_CONSTANT, borderValue=255)


def _recrop_line_to_ink(gray: np.ndarray, pad_x: int = 10, pad_y: int = 6) -> np.ndarray:
    ink = _line_ink_mask(gray)
    if not ink.any():
        return gray
    ys, xs = np.where(ink)
    h, w = gray.shape[:2]
    left = max(0, int(xs.min()) - pad_x)
    right = min(w, int(xs.max()) + 1 + pad_x)
    top = max(0, int(ys.min()) - pad_y)
    bottom = min(h, int(ys.max()) + 1 + pad_y)
    if right <= left or bottom <= top:
        return gray
    return gray[top:bottom, left:right]


# ## straighten_line_crop: small-angle deskew on a line crop when PCA says it helps.

def straighten_line_crop(crop: Image.Image) -> tuple[Image.Image, dict]:
    meta = {"enabled": LINE_STRAIGHTENING_ENABLED, "method": "line_affine_v1", "applied": False}
    if not LINE_STRAIGHTENING_ENABLED:
        return crop, meta
    gray = np.array(crop.convert("L"))
    angle, confidence = estimate_line_skew_angle(gray)
    meta.update({"estimated_angle_deg": angle, "confidence": confidence})
    if abs(angle) < LINE_STRAIGHTENING_MIN_DEG or abs(angle) > LINE_STRAIGHTENING_MAX_DEG or confidence < 0.35:
        meta["reason"] = "angle_or_confidence_outside_guardrails"
        return crop, meta
    original_score = _line_straightening_score(gray)
    best = gray
    best_angle = 0.0
    best_score = original_score
    # PCA gives the observed slope; try both signs and keep only measurable improvement.
    for candidate_angle in (-angle, angle):
        rotated = _rotate_expand_white(gray, candidate_angle)
        recropped = _recrop_line_to_ink(rotated)
        score = _line_straightening_score(recropped)
        if score < best_score:
            best = recropped
            best_angle = float(candidate_angle)
            best_score = float(score)
    meta.update({"score_before": float(original_score), "score_after": float(best_score), "applied_angle_deg": best_angle})
    if best_angle == 0.0 or best_score > original_score - 0.002:
        meta["reason"] = "no_score_improvement"
        return crop, meta
    meta["applied"] = True
    meta["padding_strategy"] = "prepad_expand_rotate_recrop"
    return Image.fromarray(best).convert("RGB"), meta


# ## crop_line: band from cleaned page, then edge cleanup + straighten, save file + metadata.

def crop_line(cleaned_page_path: Path, bbox: list[int] | tuple[int, int, int, int], output_path: Path, pad_x: int = 8, pad_y: int = FINAL_CROP_PAD_Y_PX, top_limit: int | None = None, bottom_limit: int | None = None) -> tuple[list[int], dict]:
    image = Image.open(cleaned_page_path).convert("RGB")
    width, height = image.size
    left, top, right, bottom = [int(round(float(value))) for value in bbox]
    raw_top, raw_bottom = top, bottom
    left = max(0, left - pad_x)
    right = min(width, right + pad_x)
    top = max(0, top - pad_y)
    bottom = min(height, bottom + pad_y)
    neighbor_clamp = {"enabled": top_limit is not None or bottom_limit is not None, "top_limit": top_limit, "bottom_limit": bottom_limit, "applied_top": False, "applied_bottom": False}
    if top_limit is not None:
        clamped_top = max(top, int(round(top_limit)))
        if clamped_top != top:
            neighbor_clamp["applied_top"] = True
        top = clamped_top
    if bottom_limit is not None:
        clamped_bottom = min(bottom, int(round(bottom_limit)))
        if clamped_bottom != bottom:
            neighbor_clamp["applied_bottom"] = True
        bottom = clamped_bottom
    if right <= left or bottom <= top:
        # Fall back to the raw detection band if midpoint clamping would collapse a very tight line.
        top = max(0, raw_top)
        bottom = min(height, raw_bottom)
        neighbor_clamp["fallback"] = "raw_detection_box_after_invalid_clamp"
    if right <= left or bottom <= top:
        raise ValueError(f"Invalid crop bbox: {bbox}")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    crop = image.crop((left, top, right, bottom))
    crop, edge_cleanup = decontaminate_crop_edges(crop, pad_y)
    crop, line_straightening = straighten_line_crop(crop)
    crop.save(output_path)
    return [left, top, right, bottom], {"output_path": str(output_path), "bbox": [left, top, right, bottom], "pad_x": pad_x, "pad_y": pad_y, "neighbor_clamp": neighbor_clamp, "edge_decontamination": edge_cleanup, "line_straightening": line_straightening}


# ## assign_group_splits: shuffle pages into train/val/test for exported rows.

def assign_group_splits(rows: list[dict], seed: int = 42, train_ratio: float = 0.70, val_ratio: float = 0.15) -> list[dict]:
    pages = sorted({row["page_id"] for row in rows})
    rng = random.Random(seed)
    rng.shuffle(pages)
    n = len(pages)
    if n >= 20:
        val_n = max(10, round(n * val_ratio))
        test_n = max(10, round(n * (1.0 - train_ratio - val_ratio)))
    else:
        val_n = max(1, round(n * val_ratio)) if n > 2 else 0
        test_n = max(1, round(n * (1.0 - train_ratio - val_ratio))) if n > 2 else 0
    train_n = max(0, n - val_n - test_n)
    split_by_page = {page: "train" for page in pages[:train_n]}
    split_by_page.update({page: "val" for page in pages[train_n:train_n + val_n]})
    split_by_page.update({page: "test" for page in pages[train_n + val_n:]})
    for row in rows:
        row["dataset_split"] = split_by_page.get(row["page_id"], "train")
    return rows


def split_counts(rows: list[dict]) -> dict:
    pages_by_split = defaultdict(set)
    rows_by_split = Counter()
    for row in rows:
        split = row.get("dataset_split", "")
        rows_by_split[split] += 1
        pages_by_split[split].add(row.get("page_id"))
    out = {}
    for split in ["train", "val", "test"]:
        out[f"{split}_pages"] = len(pages_by_split[split])
        out[f"{split}_rows"] = rows_by_split[split]
    return out


## Page Input Builder


In [10]:
# ## Per paired page: extract tif + GT, preprocess, drop pages with no GT text lines.
def build_page_inputs() -> list[dict]:
    pages = []
    for page_no in paired_page_numbers:
        page_id = f"page_{page_no:04d}"
        image_zip, image_member = image_index[page_no]
        gt_zip, gt_member = gt_index[page_no]
        raw_page_path = extract_zip_member(image_zip, image_member, EXTRACTED_PAGES_DIR / f"{page_id}.tif")
        gt_text = read_gt_text(gt_zip, gt_member, EXTRACTED_GT_DIR / f"{page_id}.txt")
        gt_lines = nonempty_lines(gt_text)
        if not gt_lines:
            continue
        cleaned_page_path = CLEANED_PAGE_DIR / f"{page_id}_cleaned.png"
        metadata_path = PREPROCESSING_METADATA_DIR / f"{page_id}.json"
        metadata = preprocess_page(raw_page_path, cleaned_page_path, metadata_path)
        pages.append({
            "page_id": page_id,
            "page_no": page_no,
            "raw_page_path": str(raw_page_path),
            "gt_path": str(EXTRACTED_GT_DIR / f"{page_id}.txt"),
            "gt_lines": gt_lines,
            "ground_truth_line_count": len(gt_lines),
            "cleaned_page_path": str(cleaned_page_path),
            "preprocessing_metadata_path": str(metadata_path),
            "preprocessing": metadata,
            "source_archive": str(image_zip),
            "image_member": image_member,
            "gt_member": gt_member,
        })
    return pages

pages = build_page_inputs()
write_json(P1_OUTPUT_DIR / "page_inputs_summary.json", {"page_count": len(pages), "gt_line_count": sum(len(page["gt_lines"]) for page in pages)})
summary_table({"prepared_pages": len(pages), "gt_line_count": sum(len(page["gt_lines"]) for page in pages), "cleaned_page_dir": str(CLEANED_PAGE_DIR)}, "Prepared cleaned pages")
if not pages:
    raise RuntimeError("P1 found zero paired pages. Check that raw image and ground-truth zip files are present before running P1; refusing to write empty manifests.")


### Prepared cleaned pages

,prepared_pages,gt_line_count,cleaned_page_dir
0,784,29393,C:\Users\Jobias\OneDrive\Desktop\Project Decode\data\derived\target_cleaned_pages\projection_v1_affine_full


## Decision Records


In [11]:
# ## Baseline: one projection config per page, cache binary, write base decisions + jsonl.
def decision_from_attempt(page: dict, attempt: dict, accepted: bool, source_run_id: str, rejected_reason: str = "") -> dict:
    return {
        "page_id": page["page_id"],
        "accepted": bool(accepted),
        "selected_method": attempt["method"],
        "selected_config_name": attempt["config_name"],
        "selected_config": attempt.get("config", {}),
        "selected_boxes": attempt.get("boxes", []) if accepted else [],
        "ground_truth_line_count": int(page["ground_truth_line_count"]),
        "detected_line_count": int(attempt["detected_line_count"]),
        "line_count_delta": int(attempt["line_count_delta"]),
        "source_cleaned_page": page["cleaned_page_path"],
        "cleaned_page_run_label": CLEANED_PAGE_RUN_LABEL,
        "source_run_id": source_run_id,
        "decision_pass": int(attempt["decision_pass"]),
        "notes": attempt.get("notes", []),
        "rejected_reason": "" if accepted else rejected_reason,
        "attempt_history": attempt.get("attempt_history", []),
    }


def evaluate_page_base(page: dict) -> tuple[dict, np.ndarray]:
    binary = binary_from_cleaned_page(Path(page["cleaned_page_path"]))
    boxes = detect_boxes(binary, BASELINE_CONFIG)
    attempt = attempt_from_boxes(page, BASELINE_CONFIG, boxes, binary, "baseline_projection", 1, [])
    attempt["attempt_history"] = [{k: v for k, v in attempt.items() if k != "boxes"}]
    accepted = attempt["line_count_delta"] == 0
    decision = decision_from_attempt(page, attempt, accepted, f"{P1_RUN_ID}_base", "baseline_line_count_mismatch")
    return decision, binary

base_decisions = []
binary_by_page = {}
for idx, page in enumerate(pages, start=1):
    decision, binary = evaluate_page_base(page)
    base_decisions.append(decision)
    binary_by_page[page["page_id"]] = binary
    if idx % 100 == 0 or idx == len(pages):
        print(f"baseline evaluated {idx}/{len(pages)}")

write_jsonl(P1_OUTPUT_DIR / "base_page_crop_decisions.jsonl", base_decisions)
base_summary = {"page_count": len(base_decisions), "base_exact_pages": sum(row["accepted"] for row in base_decisions), "base_rejected_pages": sum(not row["accepted"] for row in base_decisions), "delta_histogram": dict(Counter(str(row["line_count_delta"]) for row in base_decisions))}
write_json(P1_OUTPUT_DIR / "base_summary.json", base_summary)
summary_table(base_summary, "Base projection assignment")


baseline evaluated 100/784
baseline evaluated 200/784
baseline evaluated 300/784
baseline evaluated 400/784
baseline evaluated 500/784
baseline evaluated 600/784
baseline evaluated 700/784
baseline evaluated 784/784


### Base projection assignment

,page_count,base_exact_pages,base_rejected_pages,delta_histogram
0,784,129,655,"{'-3': 95, '-2': 171, '-1': 193, '0': 129, '1': 6, '-4': 70, '-7': 15, '-26': 1, '-13': 1, '-5': 21, '-8': 6, '-6': 22, '-30': 1, '14': ..."


## Iterative Page Selector

The flow here is: one baseline assignment pass, then `P1_ITERATIONS` refinement passes.
By default `P1_ITERATIONS=2`, so a normal P1 run means baseline + refinement pass 1 + refinement pass 2.


In [12]:
# ## select_page_decision: keep accepted, else try configs, repair, cleanup; state for next iter.
# Why: repair/cleanup passes target undercount and overcount failure modes separately for stability.
def select_page_decision(page: dict, previous_decision: dict, iteration: int) -> dict:
    if previous_decision.get("accepted"):
        kept = dict(previous_decision)
        kept["source_run_id"] = f"{P1_RUN_ID}_iter{iteration:02d}"
        kept["notes"] = list(kept.get("notes", [])) + [f"preserved_from_previous_iteration={iteration-1}"]
        return kept
    binary = binary_by_page.get(page["page_id"])
    if binary is None:
        binary = binary_from_cleaned_page(Path(page["cleaned_page_path"]))
        binary_by_page[page["page_id"]] = binary
    attempts = []
    for cfg in candidate_configs(iteration):
        boxes = detect_boxes(binary, cfg)
        method = "baseline_projection" if cfg["name"] == BASELINE_CONFIG["name"] else "low_threshold_projection"
        decision_pass = 1 if method == "baseline_projection" else 2
        attempts.append(attempt_from_boxes(page, cfg, boxes, binary, method, decision_pass, []))
    exact = exact_attempt(attempts)
    attempt_history = [{k: v for k, v in attempt.items() if k != "boxes"} for attempt in attempts]
    if exact:
        exact["attempt_history"] = attempt_history
        return decision_from_attempt(page, exact, True, f"{P1_RUN_ID}_iter{iteration:02d}")
    carried = best_near_attempt(attempts)
    repair = undercount_repair_attempt(page, carried, binary)
    if repair:
        attempt_history.append({k: v for k, v in repair.items() if k != "boxes"})
        repair["attempt_history"] = attempt_history
        return decision_from_attempt(page, repair, True, f"{P1_RUN_ID}_iter{iteration:02d}")
    cleanup = overcount_cleanup_attempt(page, carried, binary)
    if cleanup:
        attempt_history.append({k: v for k, v in cleanup.items() if k != "boxes"})
        cleanup["attempt_history"] = attempt_history
        return decision_from_attempt(page, cleanup, True, f"{P1_RUN_ID}_iter{iteration:02d}")
    carried["attempt_history"] = attempt_history
    return decision_from_attempt(page, carried, False, f"{P1_RUN_ID}_iter{iteration:02d}", "no_exact_count_candidate")

page_by_id = {page["page_id"]: page for page in pages}
current_decisions = {decision["page_id"]: decision for decision in base_decisions}
iteration_summaries = []


## P1 Refinement Loop

This cell is the iterative part of P1.
The previous code cell defines `select_page_decision(...)` and seeds `current_decisions` from the baseline pass.
Running the code cell below performs the refinement passes and writes the per-iteration plus final outputs.


In [ ]:
# ## Refinement loop: repeat assignment on rejected pages; jsonl + summary per iteration.
print(f"Starting P1 refinement loop: baseline already completed, now running {P1_ITERATIONS} refinement pass(es).")
for iteration in range(1, P1_ITERATIONS + 1):
    print(f"=== P1 refinement pass {iteration}/{P1_ITERATIONS} ===")
    next_decisions = {}
    for idx, page in enumerate(pages, start=1):
        next_decisions[page["page_id"]] = select_page_decision(page, current_decisions[page["page_id"]], iteration)
        if idx % 100 == 0 or idx == len(pages):
            print(f"iteration {iteration}: evaluated {idx}/{len(pages)}")
    rows = [next_decisions[page["page_id"]] for page in pages]
    iter_summary = {
        "iteration": iteration,
        "accepted_pages": sum(row["accepted"] for row in rows),
        "rejected_pages": sum(not row["accepted"] for row in rows),
        "new_recoveries": sum(next_decisions[pid].get("accepted") and not current_decisions[pid].get("accepted") for pid in next_decisions),
        "method_counts": dict(Counter(row["selected_method"] for row in rows if row["accepted"])),
        "delta_histogram": dict(Counter(str(row["line_count_delta"]) for row in rows)),
    }
    iteration_summaries.append(iter_summary)
    write_jsonl(P1_OUTPUT_DIR / f"page_crop_decisions_iter{iteration:02d}.jsonl", rows)
    write_json(P1_OUTPUT_DIR / f"summary_iter{iteration:02d}.json", iter_summary)
    current_decisions = next_decisions

final_decisions = [current_decisions[page["page_id"]] for page in pages]
write_jsonl(P1_OUTPUT_DIR / "page_crop_decisions.jsonl", final_decisions)
write_json(P1_OUTPUT_DIR / "iteration_summaries.json", {"iterations": iteration_summaries})
display(pd.DataFrame(iteration_summaries))


Starting P1 refinement loop: baseline already completed, now running 2 refinement pass(es).
=== P1 refinement pass 1/2 ===
iteration 1: evaluated 100/784
iteration 1: evaluated 200/784


## Manifest And Crop Writer


In [ ]:
# ## Manifests: emit line jsonl from final decisions; base and final run labels + summary json.
# Why: page-level acceptance preserves within-page ordering, which is important for robust CER/WER evaluation.
def build_manifest_rows(decisions: list[dict], *, crop_dir: Path, accepted_page_cap: int = 0) -> tuple[list[dict], list[dict]]:
    crop_dir = clear_generated_dir(crop_dir)
    rows = []
    rejected = []
    accepted_pages = 0
    for decision in decisions:
        page = page_by_id[decision["page_id"]]
        if not decision.get("accepted"):
            rejected.append({
                "page_id": decision["page_id"],
                "reason": "line_count_mismatch_after_assignment",
                "detected_line_count": decision.get("detected_line_count"),
                "ground_truth_line_count": decision.get("ground_truth_line_count"),
                "line_count_delta": decision.get("line_count_delta"),
                "selected_method": decision.get("selected_method"),
                "selected_config_name": decision.get("selected_config_name"),
                "rejected_reason": decision.get("rejected_reason"),
                "preprocessed_page_path": page["cleaned_page_path"],
                "source_archive": page["source_archive"],
                "image_member": page["image_member"],
                "gt_member": page["gt_member"],
            })
            continue
        if accepted_page_cap and accepted_pages >= accepted_page_cap:
            continue
        boxes = decision.get("selected_boxes") or []
        gt_lines = page["gt_lines"]
        if len(boxes) != len(gt_lines):
            rejected.append({"page_id": decision["page_id"], "reason": "accepted_decision_box_count_drift", "detected_line_count": len(boxes), "ground_truth_line_count": len(gt_lines)})
            continue
        accepted_pages += 1
        page_row_start = len(rows)
        for order, (box, gt_line) in enumerate(zip(boxes, gt_lines), start=1):
            line_id = f"{page['page_id']}_line_{order:04d}"
            crop_path = crop_dir / f"{line_id}.png"
            prev_box = boxes[order - 2] if order > 1 else None
            next_box = boxes[order] if order < len(boxes) else None
            top_limit = int(round((float(prev_box[3]) + float(box[1])) / 2.0)) if prev_box else None
            bottom_limit = int(round((float(box[3]) + float(next_box[1])) / 2.0)) if next_box else None
            clipped, crop_meta = crop_line(Path(page["cleaned_page_path"]), box, crop_path, top_limit=top_limit, bottom_limit=bottom_limit)
            normalized = normalize_text(gt_line)
            crop_quality = crop_quality_audit(crop_path, normalized)
            rows.append({
                "sample_id": f"target_{line_id}",
                "source_dataset": "target",
                "page_id": page["page_id"],
                "group_id": page["page_id"],
                "line_id": line_id,
                "image_path": str(crop_path),
                "source_image_path": page["raw_page_path"],
                "preprocessed_page_path": page["cleaned_page_path"],
                "preprocessing_metadata_path": page["preprocessing_metadata_path"],
                "raw_text": gt_line,
                "normalized_text": normalized,
                "dataset_split": "",
                "page_alignment_mode": "exact_pagewise_assignment",
                "page_acceptance_tier": "exact",
                "gt_alignment_source": "exact_order",
                "force_train_split": False,
                "detected_line_count": decision["detected_line_count"],
                "ground_truth_line_count": decision["ground_truth_line_count"],
                "line_count_delta": decision["line_count_delta"],
                "bbox": clipped,
                "sentence_width_px": clipped[2] - clipped[0],
                "sentence_height_px": clipped[3] - clipped[1],
                "sentence_top_px": clipped[1],
                "sentence_bottom_px": clipped[3],
                "source_archive": page["source_archive"],
                "source_xml": "",
                "crop_quality": crop_quality,
                "line_trainable": crop_quality["status"] == "ok",
                "page_preprocessing": page["preprocessing"],
                "crop_cleaning": crop_meta,
                "p1_assignment_run_id": P1_RUN_ID,
                "p1_decisions_path": str(P1_OUTPUT_DIR / "page_crop_decisions.jsonl"),
                "p1_selected_method": decision.get("selected_method", ""),
                "p1_selected_config_name": decision.get("selected_config_name", ""),
                "p1_decision_pass": decision.get("decision_pass"),
                "warnings": [] if crop_quality["status"] == "ok" else [{"type": "crop_quality_flag", "reasons": crop_quality["reasons"]}],
            })
        page_rows = rows[page_row_start:]
        page_clean = all(row["crop_quality"]["status"] == "ok" for row in page_rows)
        for row in page_rows:
            row["page_clean"] = page_clean
    return assign_group_splits(rows, seed=SEED), rejected

base_rows, base_rejected = build_manifest_rows(base_decisions, crop_dir=BASE_CROP_DIR, accepted_page_cap=0)
base_manifest_path = MANIFESTS_DIR / f"target_lines_{BASE_RUN_LABEL}.jsonl"
base_rejected_path = MANIFESTS_DIR / f"target_rejected_pages_{BASE_RUN_LABEL}.jsonl"
write_jsonl(base_manifest_path, base_rows)
write_jsonl(base_rejected_path, base_rejected)

final_rows, final_rejected = build_manifest_rows(final_decisions, crop_dir=FINAL_CROP_DIR, accepted_page_cap=TARGET_ACCEPTED_PAGES)
final_manifest_path = MANIFESTS_DIR / f"target_lines_{DATA_RUN_LABEL}.jsonl"
final_rejected_path = MANIFESTS_DIR / f"target_rejected_pages_{DATA_RUN_LABEL}.jsonl"
write_jsonl(final_manifest_path, final_rows)
write_jsonl(final_rejected_path, final_rejected)

summary = {
    "run_label": DATA_RUN_LABEL,
    "base_run_label": BASE_RUN_LABEL,
    "p1_run_id": P1_RUN_ID,
    "paired_pages": len(pages),
    "base_accepted_pages": len({row["page_id"] for row in base_rows}),
    "final_accepted_pages": len({row["page_id"] for row in final_rows}),
    "final_rejected_pages": len(final_rejected),
    "final_line_count": len(final_rows),
    "final_trainable_line_count": sum(1 for row in final_rows if row.get("line_trainable")),
    "final_page_clean_pages": len({row["page_id"] for row in final_rows if row.get("page_clean")}),
    "crop_quality_reason_counts": dict(Counter(reason for row in final_rows for reason in row.get("crop_quality", {}).get("reasons", []))),
    "split_counts": split_counts(final_rows),
    "manifest_path": str(final_manifest_path),
    "rejected_path": str(final_rejected_path),
    "base_manifest_path": str(base_manifest_path),
    "base_rejected_path": str(base_rejected_path),
    "decision_manifest_path": str(P1_OUTPUT_DIR / "page_crop_decisions.jsonl"),
    "cleaned_page_dir": str(CLEANED_PAGE_DIR),
    "crop_dir": str(FINAL_CROP_DIR),
}
write_json(MANIFESTS_DIR / f"target_summary_{DATA_RUN_LABEL}.json", summary)
write_json(P1_OUTPUT_DIR / "final_summary.json", summary)
summary_table(summary, "Final P1 assignment summary")
display_paths([base_manifest_path, base_rejected_path, final_manifest_path, final_rejected_path, P1_OUTPUT_DIR / "page_crop_decisions.jsonl", FINAL_CROP_DIR], "Generated artifacts")


## Contact Sheet And Rejection Export


In [ ]:
# ## QA: random contact sheet PNG; rejected subset jsonl for spot checks.
def make_contact_sheet(rows: list[dict], output_path: Path, max_items: int = 48, seed: int = 42):
    if not rows:
        return None
    rng = random.Random(seed)
    sample = list(rows)
    rng.shuffle(sample)
    sample = sample[:max_items]
    thumb_w, thumb_h = 360, 96
    label_h = 44
    cols = 3
    rows_count = math.ceil(len(sample) / cols)
    sheet = Image.new("RGB", (cols * thumb_w, rows_count * (thumb_h + label_h)), "white")
    draw = ImageDraw.Draw(sheet)
    for idx, row in enumerate(sample):
        x = (idx % cols) * thumb_w
        y = (idx // cols) * (thumb_h + label_h)
        image = Image.open(row["image_path"]).convert("RGB")
        image.thumbnail((thumb_w - 10, thumb_h - 10))
        sheet.paste(image, (x + 5, y + 5))
        label = f"{row['line_id']} | {row.get('p1_selected_method','')} | {row['normalized_text'][:52]}"
        draw.text((x + 5, y + thumb_h + 4), label, fill=(0, 0, 0))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    sheet.save(output_path)
    return output_path

contact_sheet_path = make_contact_sheet(final_rows, P1_OUTPUT_DIR / "target_contact_sheet.png")
if contact_sheet_path:
    display_paths([contact_sheet_path], "Contact sheet")
    display(Markdown(f"![target contact sheet]({contact_sheet_path})"))

rejected_subset_path = P1_OUTPUT_DIR / "rejected_page_subset.jsonl"
write_jsonl(rejected_subset_path, final_rejected)
display_paths([rejected_subset_path], "Rejected subset for follow-up inspection")


## Takeaway

This notebook optimizes crop assignment, not OCR accuracy. The final manifest is safe for P2 only after the contact sheet and rejection subset look sane.